<!-- notebook:description -->
# Notebook: RQ1 — Oracle Analysis

**Rôle** : analyser la validation des oracles générés pour RQ1 à partir des rapports expérimentaux.

**Objectif** : produire des tables et figures permettant d'évaluer précision, complétude et patterns d'échec des oracles.

*Notebook basé sur données réelles (dernier rapport JSON RQ1). Pas de simulation.*

## Énoncé de la question de recherche (RQ1)
- **RQ1 — Oracle Generation** : Dans quelle mesure un agent IA peut-il générer automatiquement des oracles précis et complets à partir de documentation d'API ?

> Notebook basé sur données réelles (rapports JSON générés par l’expérimentation RQ1).

## Load latest RQ1 report

In [4]:
# --- cell-doc ---
# But: exécuter un bloc de l'analyse (doc-only).
# Entrées: variables définies par les cellules précédentes.
# Sorties: variables/figures/fichiers produits.

import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)

# Ensure repo root is importable (enables `import experiments.*`)
PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.reporting_utils import PublicationStyle, apply_publication_style, save_figure

# Apply a consistent publication style for plots
apply_publication_style(
    PublicationStyle(
        seaborn_style="whitegrid",
        seaborn_context="paper",
        font_scale=1.1,
        palette="colorblind",
    )
)

# Define directories for experiment results and figures
EXPERIMENTS_RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"
EXPERIMENTS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = EXPERIMENTS_RESULTS_DIR / "figures" / "rq1"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, stem: str):
    """Save a figure to FIGURES_DIR as PNG+PDF (stem without extension)."""
    return save_figure(fig, FIGURES_DIR / stem, formats=["png", "pdf"], dpi=300)

# Locate the latest RQ1 report file
REPORT_DIR = EXPERIMENTS_RESULTS_DIR / "rq1"
report_files = sorted(REPORT_DIR.glob("rq1_report_*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
if not report_files:
    raise FileNotFoundError(
        f"No RQ1 report found in {REPORT_DIR.resolve()}. Run experiments/rq1_oracle_validation.py to generate rq1_report_*.json"
    )
REPORT_PATH = report_files[0]
print("Using report:", REPORT_PATH)

# Load the report data
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
report.keys()

FileNotFoundError: No RQ1 report found in /Users/aurelikama/Documents/Projet/contract-test-generation-from-api-documentation/experiments/results/rq1. Run experiments/rq1_oracle_validation.py to generate rq1_report_*.json

## Build endpoint-level metrics table

In [ ]:
# --- cell-doc ---
# But: exécuter un bloc de l'analyse (doc-only).
# Entrées: variables définies par les cellules précédentes.
# Sorties: variables/figures/fichiers produits.

rows = []
for er in report.get('endpoint_results', []):
    endpoint_id = er.get('endpoint_id')
    endpoint_name = er.get('endpoint_name')
    exec_times = er.get('execution_times', {}) or {}
    errors = er.get('errors', {}) or {}
    metrics = er.get('metrics', {}) or {}
    for model, m in metrics.items():
        rows.append({
            'endpoint_id': endpoint_id,
            'endpoint_name': endpoint_name,
            'llm_model': model,
            'precision': m.get('precision'),
            'recall': m.get('recall'),
            'f1_score': m.get('f1_score'),
            'completeness_score': m.get('completeness_score'),
            'true_positives': m.get('true_positives'),
            'false_positives': m.get('false_positives'),
            'false_negatives': m.get('false_negatives'),
            'execution_time_s': exec_times.get(model),
            'error': errors.get(model),
        })

endpoint_metrics_df = pd.DataFrame(rows)
endpoint_metrics_df.head(10)


## Aggregate metrics (from report + recompute sanity check)

In [ ]:
# --- cell-doc ---
# But: exécuter un bloc de l'analyse (doc-only).
# Entrées: variables définies par les cellules précédentes.
# Sorties: variables/figures/fichiers produits.

aggregate_metrics = report.get('aggregate_metrics', {}) or {}
aggregate_df = (
    pd.DataFrame.from_dict(aggregate_metrics, orient='index')
      .reset_index(names='llm_model')
      .sort_values('f1_mean', ascending=False)
)

display(aggregate_df)

if not endpoint_metrics_df.empty:
    recomputed = endpoint_metrics_df.groupby('llm_model').agg(
        precision_mean=('precision', 'mean'),
        recall_mean=('recall', 'mean'),
        f1_mean=('f1_score', 'mean'),
        completeness_mean=('completeness_score', 'mean'),
        n=('endpoint_id', 'count'),
    ).reset_index()
    display(recomputed.sort_values('f1_mean', ascending=False))


## Visualizations

In [ ]:
# 1) Mean F1 by oracle
df_oracles = pd.DataFrame(report["oracle_metrics"])
df_oracles = df_oracles.sort_values("mean_f1", ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=df_oracles, x="oracle", y="mean_f1", ax=ax)
ax.set_title("RQ1 — Mean F1 by oracle")
ax.set_xlabel("Oracle")
ax.set_ylabel("Mean F1")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()

save_fig(fig, "rq1_mean_f1_by_oracle")
plt.show()

# 2) F1 distribution by oracle
df_runs = pd.DataFrame(report["runs"])
if "oracle" in df_runs.columns and "f1" in df_runs.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.boxplot(data=df_runs, x="oracle", y="f1", ax=ax)
    ax.set_title("RQ1 — F1 distribution by oracle")
    ax.set_xlabel("Oracle")
    ax.set_ylabel("F1")
    ax.tick_params(axis="x", rotation=25)
    fig.tight_layout()

    save_fig(fig, "rq1_f1_distribution_by_oracle")
    plt.show()

## Export

In [ ]:
# --- cell-doc ---
# But: exécuter un bloc de l'analyse (doc-only).
# Entrées: variables définies par les cellules précédentes.
# Sorties: variables/figures/fichiers produits.

OUT_DIR = EXPERIMENTS_RESULTS_DIR / "exports" / "rq1"
OUT_DIR.mkdir(parents=True, exist_ok=True)

endpoint_metrics_df.to_csv(OUT_DIR / 'rq1_endpoint_metrics.csv', index=False)
aggregate_df.to_csv(OUT_DIR / 'rq1_aggregate_metrics.csv', index=False)

summary = {
    'report_path': str(REPORT_PATH),
    'experiment_id': report.get('experiment_id'),
    'total_endpoints': report.get('total_endpoints'),
    'llm_rankings': report.get('llm_rankings'),
}
(OUT_DIR / 'rq1_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

print('Wrote:', OUT_DIR)
